# Quick Review Notebook (Reviewer-friendly)

This notebook provides a lightweight, **read-only walkthrough** of the proposed pipeline artifacts:
- how segment keys are defined (`seg_key = comment_id__seg_id`),
- examples of UNSUP/RB/HYBRID outputs,
- how the fusion input is constructed,
- a few illustrative cases: **overlap**, **RB-only**, **UNSUP-only**, and **empty**.

**Tip:** This notebook uses the 200-row samples in `data/samples/` to keep it fast.

In [ ]:
from pathlib import Path
import pandas as pd
import json, ast, re
pd.set_option("display.max_colwidth", 120)

def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    candidates = [here, here.parent]
    for cand in candidates:
        if (cand / "data" / "samples").exists() and (cand / "notebooks").exists():
            return cand
    # fallback: if running from notebooks/ inside repo
    if (here / ".." / "data" / "samples").resolve().exists():
        return (here / "..").resolve()
    raise FileNotFoundError("Could not locate repo root (expected folders: data/samples and notebooks).")

REPO_ROOT = find_repo_root()
SAMPLES = REPO_ROOT / "data" / "samples"
DATA = REPO_ROOT / "data"
OUTPUTS = REPO_ROOT / "outputs"

# Load samples
seg = pd.read_csv(SAMPLES/"sample_segments_index_200.csv")
fusion = pd.read_csv(SAMPLES/"sample_fusion_input_200.csv")

unsup = pd.read_csv(SAMPLES/"sample_unsup_200.csv")
rb    = pd.read_csv(SAMPLES/"sample_rb_200.csv")
hyb   = pd.read_csv(SAMPLES/"sample_hybrid_200.csv")

print("Loaded:")
print("  segments_index sample:", seg.shape)
print("  fusion_input sample   :", fusion.shape)
print("  unsup sample          :", unsup.shape)
print("  rb sample             :", rb.shape)
print("  hybrid sample          :", hyb.shape)

def parse_list_cell(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return [str(t).strip() for t in x if str(t).strip()]
    s = str(x).strip()
    if not s or s.lower() in {"nan","none","null","[]"}:
        return []
    if s.startswith("[") and s.endswith("]"):
        try:
            v = json.loads(s)
            if isinstance(v, list):
                return [str(t).strip() for t in v if str(t).strip()]
        except Exception:
            pass
        try:
            v = ast.literal_eval(s)
            if isinstance(v, (list, tuple)):
                return [str(t).strip() for t in v if str(t).strip()]
        except Exception:
            pass
    for sep in [";", "|"]:
        if sep in s:
            return [p.strip() for p in s.split(sep) if p.strip()]
    if "," in s:
        parts = [p.strip() for p in s.split(",") if p.strip()]
        if all(len(p.split()) <= 6 for p in parts):
            return parts
    return [s]

def norm_term(t: str) -> str:
    return re.sub(r"\s+", " ", str(t)).strip().lower()

## 1) Segment key alignment (`seg_key`)

In this project, `seg_id` is **local** (can repeat across comments).
Therefore, we align everything using:

- `seg_key = comment_id__seg_id`

Below we show that all loaded artifacts contain `seg_key` and that it is unique per segment in the sample.

In [ ]:
for name, df in [("segments_index", seg), ("fusion_input", fusion), ("unsup", unsup), ("rb", rb), ("hybrid", hyb)]:
    print(name, "has seg_key:", "seg_key" in df.columns, "| unique:", df["seg_key"].nunique(), "| rows:", len(df))

# Show 5 example keys
seg[["comment_id","seg_id","seg_key"]].head(5)

## 2) Example rows from each branch output (UNSUP / RB / HYBRID)

We show 10 rows for each output (sampled files), focusing on:
- segment text,
- predicted aspect term lists (or best term).

In [ ]:
def show_cols(df, cols, n=10):
    cols = [c for c in cols if c in df.columns]
    return df[cols].head(n)

print("UNSUP example:")
display(show_cols(unsup, ["seg_key","seg_text","aspect_terms_unsup","top_aspect_terms_unsup","fallback_mode"], 10))

print("RB example:")
display(show_cols(rb, ["seg_key","seg_text","aspect_terms_rb_clean","rb_lang_used"], 10))

print("HYBRID example:")
display(show_cols(hyb, ["seg_key","seg_text","hybrid_best_term","hybrid_topk_terms","hybrid_best_score"], 10))

## 3) Fusion input structure

The fusion stage expects:

- `data/dataset_merged_with_unsup_rb_aspects_ALLCOLS.csv`

This pack constructs it by starting from the **master segment universe** (`segments_index`) and
left-joining RB and UNSUP outputs by `seg_key`. The sample below illustrates the joined columns.

In [ ]:
display(show_cols(fusion, [
    "seg_key","seg_text",
    "aspect_terms_rb_clean","aspect_terms_unsup",
    "top_aspect_terms_unsup","fallback_mode"
], 10))

## 4) Illustrative cases (overlap / RB-only / UNSUP-only / empty)

We classify each segment using the term lists:
- RB list: `aspect_terms_rb_clean`
- UNSUP list: `aspect_terms_unsup`

Cases:
- **overlap**: intersection non-empty
- **rb_only**: RB non-empty, UNSUP empty
- **unsup_only**: UNSUP non-empty, RB empty
- **empty**: both empty

In [ ]:
def classify_case(row):
    rb_terms = [norm_term(t) for t in parse_list_cell(row.get("aspect_terms_rb_clean", "")) if norm_term(t)]
    un_terms = [norm_term(t) for t in parse_list_cell(row.get("aspect_terms_unsup", "")) if norm_term(t)]
    inter = set(rb_terms) & set(un_terms)
    if inter:
        return "overlap"
    if rb_terms and not un_terms:
        return "rb_only"
    if un_terms and not rb_terms:
        return "unsup_only"
    return "empty"

fusion_case = fusion.copy()
fusion_case["case"] = fusion_case.apply(classify_case, axis=1)
fusion_case["case"].value_counts()

In [ ]:
def pick_examples(df, case, k=3):
    sub = df[df["case"]==case].copy()
    if len(sub)==0:
        return sub
    cols = [c for c in ["seg_key","seg_text","aspect_terms_rb_clean","aspect_terms_unsup","top_aspect_terms_unsup","fallback_mode"] if c in sub.columns]
    return sub[cols].head(k)

for case in ["overlap","rb_only","unsup_only","empty"]:
    print("\nCASE:", case)
    display(pick_examples(fusion_case, case, 3))

## Notes

- This notebook is intended for **inspection and sanity-checking**.
- The full hybrid logic (PMI gate, context-fit ranking, overlap bonus, fallback) is implemented in `notebooks/06_fusion_selection.ipynb`.
- For reproducible merges, always use `seg_key` (or the pair `comment_id, seg_id`).